## setup ace-step-1.5

In [2]:
%cd /content
!git clone https://github.com/ace-step/ACE-Step-1.5
%cd ACE-Step-1.5
!uv venv && uv pip install -e .

/content
Cloning into 'ACE-Step-1.5'...
remote: Enumerating objects: 12531, done.
remote: Counting objects: 100% (599/599), done.
remote: Compressing objects: 100% (185/185), done.
remote: Total 12531 (delta 495), reused 414 (delta 414), pack-reused 11932 (from 3)
Receiving objects: 100% (12531/12531), 13.07 MiB | 15.76 MiB/s, done.
Resolving deltas: 100% (8044/8044), done.
/content/ACE-Step-1.5
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate
Using Python 3.12.13 environment at: /usr
Resolved 141 packages in 2.11s
Prepared 20 packages in 4.17s
Uninstalled 7 packages in 866ms
Installed 20 packages in 247ms
 + ace-step==1.5.0 (from file:///content/ACE-Step-1.5)
 + diskcache==5.6.3
 + einx==0.4.3
 - gradio==5.50.0
 + gradio==6.2.0
 - gradio-client==1.14.0
 + gradio-client==2.0.2
 - huggingface-hub==1.11.0
 + huggingface-hub==0.36.2
 + lightning==2.6.1
 + lightning-utilities==0.15.3
 - llvmlite==0.43.0
 

## setup ace-step-ui

In [3]:
%cd /content
!git clone https://github.com/fspecii/ace-step-ui
%cd /content/ace-step-ui
!./setup.sh

/content
Cloning into 'ace-step-ui'...
remote: Enumerating objects: 605, done.
remote: Counting objects: 100% (352/352), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 605 (delta 289), reused 233 (delta 233), pack-reused 253 (from 1)
Receiving objects: 100% (605/605), 20.60 MiB | 15.05 MiB/s, done.
Resolving deltas: 100% (347/347), done.
/content/ace-step-ui
  ACE-Step UI Setup
Found ACE-Step at: ../ACE-Step-1.5
Creating .env file...

Installing frontend dependencies...
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙npm warn deprecated node-domexception@1.0.0: Use your platform's native DOMException instead
⠙⠹⠸⠼⠴⠦⠧npm warn deprecated glob@10.5.0: Old versions of glob are not supported, and contain widely publicized security vulnerabilities, which have been fixed in the current version. Please update. Support for old versions may be purchased (at exorbitant rates) by contacting i@izs.me
⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦
added 150 packages, and audited 151 packages in 6s
⠦
⠦28 packages are lo

## patch code

In [4]:
import re
from pathlib import Path

# Patch float16 (not running on colab) to float32
print("Patch FLOAT16...")
for f in Path("/content/ACE-Step-1.5").rglob("*.py"):
    try:
        text = f.read_text()
        text2 = text.replace("torch.float16", "torch.float32")

        if text != text2:
            f.write_text(text2)
            print("- ", f)
    except:
        pass

# Patch client vite config to bypass CORS
p = Path("/content/ace-step-ui/vite.config.ts")
text = p.read_text()
if "allowedHosts: true" not in text:
    text2 = re.sub(
        r"host: '0.0.0.0',",
        r"host: '0.0.0.0', strictPort: true, allowedHosts: true, cors: true, hmr: false,",
        text,
        count=1
    )
    p.write_text(text2)
    print("Patch client: " + ('done!' if text != text2 else 'fail!'))

# Patch server origin to bypass CORS
p = Path("/content/ace-step-ui/server/src/index.ts")
text = p.read_text()
if "origin: true" not in text:
    text2 = re.sub(
        r"origin: ",
        r"origin: true,/* ",
        text,
        count=1
    )
    text2 = re.sub(
        r"credentials: true",
        r" */credentials: true",
        text2,
        count=1
    )
    p.write_text(text2)
    print("Patch server: " + ('done!' if text != text2 else 'fail!'))

Patch FLOAT16...
-  /content/ACE-Step-1.5/scripts/profile_vram.py
-  /content/ACE-Step-1.5/acestep/training/trainer.py
-  /content/ACE-Step-1.5/acestep/training_v2/fixed_lora_module.py
-  /content/ACE-Step-1.5/acestep/training_v2/estimate.py
-  /content/ACE-Step-1.5/acestep/training_v2/model_loader.py
-  /content/ACE-Step-1.5/acestep/training_v2/trainer_vanilla.py
-  /content/ACE-Step-1.5/acestep/training_v2/cli/train_vanilla.py
-  /content/ACE-Step-1.5/acestep/third_parts/nano-vllm/build/lib/nanovllm/engine/model_runner.py
-  /content/ACE-Step-1.5/acestep/third_parts/nano-vllm/nanovllm/engine/model_runner.py
-  /content/ACE-Step-1.5/acestep/core/generation/handler/memory_utils.py
-  /content/ACE-Step-1.5/acestep/core/generation/handler/init_service_orchestrator.py
-  /content/ACE-Step-1.5/acestep/core/generation/handler/init_service_test.py
-  /content/ACE-Step-1.5/acestep/core/generation/handler/diffusion_test.py
Patch client: done!
Patch server: done!


## make link python3 to ace-step

In [5]:
!mkdir -p /content/ace-step-ui/ACE-Step-1.5/env/bin
!ln -s $(which python3) /content/ace-step-ui/ACE-Step-1.5/env/bin/python

## run ace-step-ui

In [ ]:
!pkill -f "npm run dev"
import os
import subprocess
import threading
from google.colab.output import eval_js

server = subprocess.Popen(
    ["npm", "run", "dev"],
    cwd="/content/ace-step-ui/server",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

client = subprocess.Popen(
    ["npm", "run", "dev"],
    cwd="/content/ace-step-ui",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

print("localhost:3000 -> " + eval_js("google.colab.kernel.proxyPort(3000)"))

def stream_output(prefix, proc):
    for line in iter(proc.stdout.readline, ''):
        print(f"{prefix} {line}", end='')

threading.Thread(
    target=stream_output,
    args=("[SERVER]", server),
    daemon=True
).start()

stream_output("[CLIENT]", client)

localhost:3000 -> https://3000-gpu-t4-s-kkb-ass1c0-vjtx9imxt50g-c.asia-southeast1-0.prod.colab.dev
[SERVER] 
[SERVER] > ace-step-ui-server@1.0.0 dev
[SERVER] > tsx watch src/index.ts
[SERVER] 
[CLIENT] 
[CLIENT] > ace-step-ui@1.0.0 dev
[CLIENT] > vite
[CLIENT] 
[CLIENT] 
[CLIENT]   VITE v6.4.1  ready in 462 ms
[CLIENT] 
[CLIENT]   ➜  Local:   http://localhost:3000/
[CLIENT]   ➜  Network: http://172.28.0.12:3000/
[SERVER] Running SQLite database migrations...
[SERVER] Migrations completed successfully!
[SERVER] ACE-Step UI Server running on http://localhost:3001
[SERVER] Environment: development
[SERVER] ACE-Step API: http://localhost:8001
[SERVER] LAN access: http://172.28.0.12:3001
